# LLM Magic Quadrant — Use-Case Recommender

**How it works:**
1. You describe your **use case** and provide **prompts** that represent real tasks
2. The notebook runs those prompts against all available models on your Databricks instance
3. Each model is scored on **Accuracy · Cost · Speed**
4. A **Magic Quadrant** is generated — showing where each model sits and which one is recommended

**Quadrant axes:**
- X → Performance Score (accuracy, higher = better)
- Y → Value Score (cost efficiency, higher = cheaper per token)
- Bubble size → Speed (larger = faster response)

**Recommendation weights** are tunable so the winner matches your priorities (quality vs cost vs speed).

---
## Cell 1 · Install & imports

In [ ]:
%pip install openai matplotlib numpy -q

In [ ]:
import os, re, ast, json, time, ssl, math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import openai

print("✓ imports ready")

---
## Cell 2 · Credentials

In [ ]:
DATABRICKS_HOST  = "https://dbc-7f2cefe3-e622.cloud.databricks.com"  # ← your workspace
DATABRICKS_TOKEN = "<YOUR-DATABRICKS-PAT>"              # ← your PAT

client = openai.OpenAI(
    base_url=f"{DATABRICKS_HOST.rstrip('/')}/serving-endpoints",
    api_key=DATABRICKS_TOKEN,
)
print("✓ client ready →", DATABRICKS_HOST)

---
## Cell 3 · Model registry

In [ ]:
# Add / remove models to match what your workspace has enabled
MODELS = [
    {"model_id": "databricks-claude-opus-4-7",             "display_name": "Claude Opus 4.7",        "provider": "Anthropic", "input_price_per_1m": 15.00, "output_price_per_1m": 75.00},
    {"model_id": "databricks-claude-sonnet-4-6",           "display_name": "Claude Sonnet 4.6",      "provider": "Anthropic", "input_price_per_1m":  3.00, "output_price_per_1m": 15.00},
    {"model_id": "databricks-claude-haiku-4-5",            "display_name": "Claude Haiku 4.5",       "provider": "Anthropic", "input_price_per_1m":  0.80, "output_price_per_1m":  4.00},
    {"model_id": "databricks-gpt-4o",                      "display_name": "GPT-4o",                 "provider": "OpenAI",    "input_price_per_1m":  2.50, "output_price_per_1m": 10.00},
    {"model_id": "databricks-gpt-4o-mini",                 "display_name": "GPT-4o-mini",            "provider": "OpenAI",    "input_price_per_1m":  0.15, "output_price_per_1m":  0.60},
    {"model_id": "databricks-gemini-2-5-pro",              "display_name": "Gemini 2.5 Pro",         "provider": "Google",    "input_price_per_1m":  1.25, "output_price_per_1m": 10.00},
    {"model_id": "databricks-gemini-2-5-flash",            "display_name": "Gemini 2.5 Flash",       "provider": "Google",    "input_price_per_1m":  0.075,"output_price_per_1m":  0.30},
    {"model_id": "databricks-llama-4-maverick",            "display_name": "Llama 4 Maverick",       "provider": "Meta",      "input_price_per_1m":  0.15, "output_price_per_1m":  0.60},
    {"model_id": "databricks-meta-llama-3-3-70b-instruct", "display_name": "Llama 3.3 70B",          "provider": "Meta",      "input_price_per_1m":  0.12, "output_price_per_1m":  0.38},
    {"model_id": "databricks-meta-llama-3-1-8b-instruct",  "display_name": "Llama 3.1 8B",           "provider": "Meta",      "input_price_per_1m":  0.02, "output_price_per_1m":  0.05},
]
PRICING = {m["model_id"]: m for m in MODELS}
print(f"✓ {len(MODELS)} models registered")

---
## Cell 4 · ⚙️ Define your use case  ← EDIT THIS

Replace the example below with your actual use case and prompts.  
Each prompt needs a `text` (what you send) and a `validator` (how to score the answer).  
Use the built-in validators or write a simple `lambda r: (label, score, detail)` for custom checks.

In [ ]:
# ── Validator helpers ─────────────────────────────────────────────────────────
def _tier(s): return "Excellent" if s>=1 else "Good" if s>=0.75 else "Partial" if s>=0.40 else "Poor"

def contains_all(*keywords):
    """All keywords must appear in response (case-insensitive)."""
    def v(r):
        t = r.lower()
        missing = [k for k in keywords if k.lower() not in t]
        score = 1 - len(missing)/len(keywords)
        return _tier(score), round(score,2), ("All keywords found" if not missing else f"Missing: {missing}")
    return v

def exact_number(expected, tolerance=0):
    """Response must contain the expected number."""
    def v(r):
        nums = re.findall(r"-?\d+(?:\.\d+)?", r.replace(",",""))
        if not nums: return "Poor", 0.0, "No number found"
        got = float(nums[0])
        diff = abs(got - expected)
        if diff <= max(tolerance, 0.01): return "Excellent", 1.0, f"Correct: {got}"
        if diff <= expected*0.1:         return "Partial",   0.4,  f"Close: {got} vs {expected}"
        return "Poor", 0.0, f"Wrong: {got} vs {expected}"
    return v

def valid_json(*required_keys):
    """Response must be valid JSON containing all required keys."""
    def v(r):
        try:
            m = re.search(r"(\{.*\}|\[.*\])", r, re.DOTALL)
            data = json.loads(m.group(1) if m else r.strip())
            if isinstance(data, list): data = data[0] if data else {}
            missing = [k for k in required_keys if k not in data]
            score = 1 - len(missing)/len(required_keys) if required_keys else 1.0
            return _tier(score), round(score,2), ("Valid JSON" if not missing else f"Missing keys: {missing}")
        except Exception as e:
            return "Poor", 0.0, f"Invalid JSON: {e}"
    return v

def run_python(*test_cases):
    """Extract Python from response, run test_cases: each is (call_string, expected_value)."""
    def v(r):
        m = re.search(r"```(?:python)?\s*\n(.*?)```", r, re.DOTALL)
        code = m.group(1).strip() if m else r.strip()
        ns = {}
        try: exec(compile(ast.parse(code),"<llm>","exec"), ns)
        except Exception as e: return "Poor", 0.0, f"Compile error: {e}"
        passed, fails = 0, []
        for call, exp in test_cases:
            try:
                res = eval(call, ns)
                if res == exp: passed += 1
                else: fails.append(f"{call}→{res!r}(exp {exp!r})")
            except Exception as e: fails.append(f"{call}→ERR:{e}")
        score = passed/len(test_cases)
        return _tier(score), round(score,2), f"{passed}/{len(test_cases)} passed" + (f": {'; '.join(fails)}" if fails else "")
    return v

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ✏️  EDIT HERE — define your use case
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

USE_CASE_NAME = "Customer Support Automation"
USE_CASE_DESC = "Classify support tickets, generate replies, and extract structured data from customer messages."

USE_CASE_PROMPTS = [
    {
        "id": "ticket_classification",
        "prompt": (
            "Classify this support ticket into one category: Billing, Technical, Shipping, Returns, Other.\n"
            "Ticket: 'My order arrived damaged and I want a refund.'\n"
            "Reply with one word only."
        ),
        "validator": contains_all("returns"),
    },
    {
        "id": "sentiment_urgency",
        "prompt": (
            "Rate the urgency of this customer message as Low, Medium, or High, and its sentiment as Positive, Neutral, or Negative.\n"
            "Message: 'This is the third time my payment has failed! I need this fixed NOW or I am cancelling!'\n"
            "Format: 'Urgency: <level>, Sentiment: <value>'"
        ),
        "validator": contains_all("high", "negative"),
    },
    {
        "id": "reply_generation",
        "prompt": (
            "Write a professional customer support reply to this message. Keep it under 60 words.\n"
            "Message: 'Where is my order? It was supposed to arrive 3 days ago.'"
        ),
        "validator": contains_all("apologize", "order"),
    },
    {
        "id": "extract_customer_info",
        "prompt": (
            "Extract customer details from this message and return as JSON with fields: name, issue, product.\n"
            "Message: 'Hi, I am Sarah. My laptop model XZ-500 keeps crashing after the last update.'\n"
            "Return only the JSON."
        ),
        "validator": valid_json("name", "issue", "product"),
    },
    {
        "id": "faq_answer",
        "prompt": (
            "Answer this customer FAQ in 1-2 sentences:\n"
            "Q: How long does a refund take to process?"
        ),
        "validator": contains_all("refund", "business days"),
    },
    {
        "id": "summarise_thread",
        "prompt": (
            "Summarise this support thread in one sentence:\n"
            "Customer: My internet is down. Agent: Have you restarted the router? "
            "Customer: Yes, still down. Agent: I will escalate to our network team."
        ),
        "validator": contains_all("internet", "escalat"),
    },
]

print(f"✓ Use case: '{USE_CASE_NAME}'")
print(f"  {len(USE_CASE_PROMPTS)} prompts defined")
for p in USE_CASE_PROMPTS:
    print(f"  · {p['id']}")

---
## ── Alternative Use Cases ──────────────────────────────────────────────────
Run **one** of the cells below to load a different use case, then continue from **Cell 5**.  
Each use case uses long, realistic prompts that stress-test reasoning, code generation, JSON extraction, and multi-step logic.

| Cell | Use Case |
|------|----------|
| **UC-A** | Financial Analysis & Earnings Intelligence |
| **UC-B** | Code Review & Security Audit |
| **UC-C** | Databricks / Data Engineering |
| **UC-D** | Legal & Compliance Document Analysis |

### UC-A · Financial Analysis & Earnings Intelligence

In [ ]:
USE_CASE_NAME = "Financial Analysis & Earnings Intelligence"
USE_CASE_DESC = (
    "Extract structured financial data, perform multi-step calculations, "
    "generate analyst commentary, identify risk factors, and classify market signals."
)

USE_CASE_PROMPTS = [
    # ── 1. Structured extraction from an earnings release ──────────────────────
    {
        "id": "earnings_extraction",
        "prompt": (
            "Extract the key financial metrics from the earnings release below and return them as a single "
            "JSON object. Required fields: revenue_usd_millions, net_income_usd_millions, eps_diluted, "
            "yoy_revenue_growth_pct, gross_margin_pct, operating_margin_pct, free_cash_flow_usd_millions.\n\n"
            "--- EARNINGS RELEASE EXCERPT ---\n"
            "Acme Corp reported Q3 2024 revenue of $4,820 million, up 14.3% year-over-year from $4,217 million "
            "in Q3 2023. Net income reached $612 million, compared with $488 million in the prior-year period. "
            "Diluted earnings per share came in at $2.47 versus $1.96 a year ago. Gross profit was $2,073 million, "
            "yielding a gross margin of 43.0%. Operating income was $741 million (operating margin: 15.4%). "
            "Free cash flow for the quarter was $524 million.\n"
            "---\n\n"
            "Return ONLY the JSON object, no explanation."
        ),
        "validator": valid_json(
            "revenue_usd_millions", "net_income_usd_millions", "eps_diluted",
            "yoy_revenue_growth_pct", "gross_margin_pct", "operating_margin_pct",
            "free_cash_flow_usd_millions"
        ),
    },

    # ── 2. Multi-step CAGR calculation ─────────────────────────────────────────
    {
        "id": "cagr_calculation",
        "prompt": (
            "A company reported the following annual revenue figures (USD millions):\n"
            "  2018: 1,200\n  2019: 1,410\n  2020: 1,350\n  2021: 1,740\n  2022: 2,100\n  2023: 2,560\n\n"
            "Calculate the Compound Annual Growth Rate (CAGR) from 2018 to 2023 using the formula:\n"
            "  CAGR = (End Value / Start Value)^(1 / Years) - 1\n\n"
            "Round your answer to two decimal places and express as a percentage (e.g., 13.42).\n"
            "Rules: return ONLY the percentage number, no % sign, no words, no formula. "
            "Example output: 13.42"
        ),
        "validator": exact_number(16.36, tolerance=0.5),
    },

    # ── 3. DCF valuation commentary ────────────────────────────────────────────
    {
        "id": "dcf_commentary",
        "prompt": (
            "You are a sell-side equity analyst. Based on the assumptions below, write a professional "
            "DCF valuation commentary paragraph (100-150 words) suitable for an equity research report. "
            "Include the implied intrinsic value per share and note key sensitivity risks.\n\n"
            "DCF Assumptions:\n"
            "  - 5-year projected free cash flows (USD M): 520, 590, 670, 755, 845\n"
            "  - Terminal growth rate: 3.5%\n"
            "  - WACC: 9.2%\n"
            "  - Terminal value discounted: 6,840 USD M\n"
            "  - Enterprise value: 10,120 USD M\n"
            "  - Net debt: 320 USD M\n"
            "  - Shares outstanding: 248 million\n"
            "  - Implied share price: $39.52\n\n"
            "Write the commentary now."
        ),
        "validator": contains_all("39.52", "wacc", "terminal", "sensitivity"),
    },

    # ── 4. Risk factor identification ──────────────────────────────────────────
    {
        "id": "risk_identification",
        "prompt": (
            "Read the financial risk disclosure below. Identify exactly 5 distinct risk factors. "
            "For each risk, provide: (1) a short risk title (3-5 words), (2) the risk category "
            "(Market, Credit, Operational, Regulatory, or Liquidity), and (3) one sentence explaining "
            "the impact. Return the result as a JSON array with fields: title, category, impact.\n\n"
            "--- RISK DISCLOSURE ---\n"
            "Our business is subject to significant exposure to interest rate fluctuations. A 100 basis "
            "point increase in benchmark rates would increase our annual interest expense by approximately "
            "$42 million. We hold $1.2 billion in accounts receivable from customers in the telecom sector, "
            "where two counterparties represent 38% of the balance. Our manufacturing operations in Southeast "
            "Asia depend on just-in-time logistics; any port disruption could halt production within 72 hours. "
            "We are currently under investigation by the EU Competition Authority regarding alleged price-fixing "
            "in our industrial components segment; fines could reach 10% of affected revenue. Our revolving "
            "credit facility of $600 million expires in 14 months and refinancing risk has increased given "
            "current credit spreads.\n"
            "---\n\n"
            "Return ONLY the JSON array."
        ),
        "validator": valid_json("title", "category", "impact"),
    },

    # ── 5. Earnings call sentiment + guidance classification ───────────────────
    {
        "id": "guidance_classification",
        "prompt": (
            "Analyze the CFO statement from an earnings call below. Classify:\n"
            "  (a) Overall tone: Bullish / Neutral / Bearish\n"
            "  (b) Revenue guidance direction: Raised / Maintained / Lowered / Not provided\n"
            "  (c) Margin outlook: Expanding / Stable / Compressing / Unclear\n"
            "  (d) Management confidence level: High / Medium / Low\n\n"
            "--- CFO STATEMENT ---\n"
            "We are pleased to raise our full-year revenue outlook to a range of $19.4 to $19.8 billion, "
            "up from our prior guidance of $18.9 to $19.3 billion, reflecting stronger-than-expected "
            "enterprise demand in North America and APAC. On the margin front, we do anticipate some near-term "
            "headwinds from elevated cloud infrastructure costs and ongoing investment in our AI platform — we "
            "expect operating margins to step down modestly by 50 to 80 basis points in Q4 before recovering "
            "in the first half of next year. Despite these short-term pressures, we remain confident in our "
            "long-term margin expansion thesis and our ability to grow free cash flow at a double-digit rate "
            "through 2026.\n"
            "---\n\n"
            "Return the result as JSON with fields: tone, revenue_guidance, margin_outlook, confidence."
        ),
        "validator": valid_json("tone", "revenue_guidance", "margin_outlook", "confidence"),
    },

    # ── 6. Comparative financial ratio analysis ────────────────────────────────
    {
        "id": "ratio_analysis",
        "prompt": (
            "Given the financial data for two competing companies, calculate the following ratios for each "
            "and recommend which company is the better investment from a value perspective. "
            "Include P/E, P/S, EV/EBITDA, and Debt/Equity.\n\n"
            "Company A — TechFlow Inc:\n"
            "  Share price: $84.50 | Shares outstanding: 310M | Net income: $1,240M\n"
            "  Revenue: $8,900M | EBITDA: $2,100M | Total debt: $3,200M | Cash: $1,100M\n"
            "  Equity (book): $6,800M\n\n"
            "Company B — DataWave Corp:\n"
            "  Share price: $42.20 | Shares outstanding: 520M | Net income: $640M\n"
            "  Revenue: $5,200M | EBITDA: $980M | Total debt: $1,400M | Cash: $800M\n"
            "  Equity (book): $4,100M\n\n"
            "Show your calculations, then state your recommendation and rationale in 2-3 sentences."
        ),
        "validator": contains_all("p/e", "ev/ebitda", "debt", "recommend"),
    },
]

print(f"✓ Use case: '{USE_CASE_NAME}'")
print(f"  {len(USE_CASE_PROMPTS)} prompts  |  financial extraction, CAGR, DCF, risk ID, guidance, ratios")

### UC-B · Code Review & Security Audit

In [ ]:
USE_CASE_NAME = "Code Review & Security Audit"
USE_CASE_DESC = (
    "Detect bugs, identify security vulnerabilities, generate unit tests, "
    "analyse algorithmic complexity, and refactor code for production quality."
)

USE_CASE_PROMPTS = [
    # ── 1. Bug detection in a Python function ──────────────────────────────────
    {
        "id": "bug_detection",
        "prompt": (
            "Review the Python function below. Identify ALL bugs (logic errors, off-by-one errors, "
            "edge cases not handled, incorrect operations). For each bug: state the line/issue, "
            "explain why it is wrong, and provide the corrected code. Then provide a fully corrected "
            "version of the function at the end.\n\n"
            "```python\n"
            "def process_transactions(transactions):\n"
            "    \"\"\"Return total, average, and list of transactions above the average.\"\"\"\n"
            "    total = 0\n"
            "    for t in transactions:\n"
            "        total += t\n"
            "    average = total / len(transactions)  # BUG 1 hidden here\n"
            "    above_avg = []\n"
            "    for i in range(1, len(transactions)):  # BUG 2 hidden here\n"
            "        if transactions[i] > average:\n"
            "            above_avg.append(transactions[i])\n"
            "    running = []\n"
            "    acc = 0\n"
            "    for t in transactions:\n"
            "        acc += t\n"
            "        running.append(acc)\n"
            "    return {\n"
            "        'total': total,\n"
            "        'average': average,\n"
            "        'above_average': above_avg,\n"
            "        'running_totals': running,\n"
            "        'max_transaction': max(transactions),  # BUG 3 hidden here\n"
            "        'count': len(above_avg) / len(transactions)  # BUG 4 hidden here\n"
            "    }\n"
            "```\n\n"
            "Note: BUG 1 = division by zero on empty list; BUG 2 = off-by-one (misses index 0); "
            "BUG 3 = crashes on empty list; BUG 4 = count should be integer not ratio."
        ),
        "validator": contains_all("empty", "off-by-one", "division", "zero"),
    },

    # ── 2. SQL injection vulnerability in a web handler ────────────────────────
    {
        "id": "sql_injection_audit",
        "prompt": (
            "Perform a security audit of the Python/Flask endpoint below. "
            "Identify every security vulnerability (including but not limited to injection flaws, "
            "authentication issues, information disclosure, and missing input validation). "
            "For each vulnerability: name it, assign a CVSS severity (Critical/High/Medium/Low), "
            "explain the attack vector, and provide the secure remediation code.\n\n"
            "```python\n"
            "from flask import Flask, request, jsonify\n"
            "import sqlite3, os\n\n"
            "app = Flask(__name__)\n"
            "DB = 'users.db'\n\n"
            "@app.route('/login', methods=['POST'])\n"
            "def login():\n"
            "    username = request.form.get('username')\n"
            "    password = request.form.get('password')\n"
            "    conn = sqlite3.connect(DB)\n"
            "    query = f\"SELECT * FROM users WHERE username='{username}' AND password='{password}'\"\n"
            "    user = conn.execute(query).fetchone()\n"
            "    if user:\n"
            "        token = os.urandom(8).hex()\n"
            "        return jsonify({'token': token, 'user_data': str(user)})\n"
            "    return jsonify({'error': f'Login failed for user {username}'}), 401\n\n"
            "@app.route('/admin/users')\n"
            "def list_users():\n"
            "    conn = sqlite3.connect(DB)\n"
            "    rows = conn.execute('SELECT * FROM users').fetchall()\n"
            "    return jsonify({'users': [str(r) for r in rows]})\n"
            "```\n"
        ),
        "validator": contains_all("sql injection", "parameterized", "authentication", "information disclosure"),
    },

    # ── 3. Generate comprehensive unit tests ───────────────────────────────────
    {
        "id": "unit_test_generation",
        "prompt": (
            "Write a comprehensive pytest test suite for the function below. "
            "Cover: happy path, edge cases (empty input, single element, negative numbers, "
            "floats, very large input), and error conditions. Use parametrize where appropriate. "
            "Each test must have a descriptive name and a one-line docstring explaining what it validates.\n\n"
            "```python\n"
            "def calculate_statistics(numbers: list[float]) -> dict:\n"
            "    \"\"\"\n"
            "    Returns a dict with: mean, median, std_dev, min, max, range, count.\n"
            "    Raises ValueError if the input list is empty.\n"
            "    Raises TypeError if any element is not numeric.\n"
            "    \"\"\"\n"
            "    if not numbers:\n"
            "        raise ValueError('Input list cannot be empty')\n"
            "    for n in numbers:\n"
            "        if not isinstance(n, (int, float)):\n"
            "            raise TypeError(f'Non-numeric value: {n}')\n"
            "    n = len(numbers)\n"
            "    mean = sum(numbers) / n\n"
            "    sorted_nums = sorted(numbers)\n"
            "    mid = n // 2\n"
            "    median = sorted_nums[mid] if n % 2 else (sorted_nums[mid-1] + sorted_nums[mid]) / 2\n"
            "    variance = sum((x - mean)**2 for x in numbers) / n\n"
            "    return {\n"
            "        'mean': mean, 'median': median, 'std_dev': variance**0.5,\n"
            "        'min': min(numbers), 'max': max(numbers),\n"
            "        'range': max(numbers) - min(numbers), 'count': n\n"
            "    }\n"
            "```\n"
        ),
        "validator": contains_all("def test_", "pytest", "valueerror", "parametrize"),
    },

    # ── 4. Algorithm complexity + optimisation ─────────────────────────────────
    {
        "id": "complexity_optimisation",
        "prompt": (
            "Analyse the time and space complexity of the function below. "
            "Then rewrite it to achieve O(n) time complexity (currently O(n²)). "
            "Explain the optimisation technique used and verify your solution produces "
            "the same output for the test cases: "
            "find_pairs([2,7,4,5,3,8], 10) → [(2,8),(7,3),(5,5)] (order may vary) and "
            "find_pairs([1,2,3], 10) → [].\n\n"
            "```python\n"
            "def find_pairs(nums, target_sum):\n"
            "    \"\"\"Return all unique pairs (a, b) where a + b == target_sum and a <= b.\"\"\"\n"
            "    pairs = []\n"
            "    for i in range(len(nums)):\n"
            "        for j in range(i + 1, len(nums)):\n"
            "            if nums[i] + nums[j] == target_sum:\n"
            "                pair = (min(nums[i], nums[j]), max(nums[i], nums[j]))\n"
            "                if pair not in pairs:\n"
            "                    pairs.append(pair)\n"
            "    return pairs\n"
            "```\n\n"
            "Provide: (1) current complexity analysis, (2) optimised implementation in a Python code block, "
            "(3) explanation of the hash set approach."
        ),
        "validator": run_python(
            ("sorted(find_pairs([2,7,4,5,3,8], 10))", sorted([(2,8),(3,7),(5,5)])),
            ("find_pairs([1,2,3], 10)", []),
        ),
    },

    # ── 5. Docker / infrastructure security misconfiguration review ────────────
    {
        "id": "infra_security_review",
        "prompt": (
            "Audit the docker-compose.yml below for security misconfigurations. "
            "For each issue found: assign a severity (Critical / High / Medium / Low), "
            "describe the risk, and provide the corrected configuration snippet.\n\n"
            "```yaml\n"
            "version: '3.8'\n"
            "services:\n"
            "  api:\n"
            "    image: myapp:latest\n"
            "    ports:\n"
            "      - '0.0.0.0:5432:5432'\n"
            "    environment:\n"
            "      - DB_PASSWORD=admin123\n"
            "      - SECRET_KEY=supersecretkey\n"
            "      - DEBUG=true\n"
            "    volumes:\n"
            "      - /:/host_root\n"
            "    privileged: true\n"
            "    user: root\n"
            "  db:\n"
            "    image: postgres:latest\n"
            "    environment:\n"
            "      - POSTGRES_PASSWORD=admin123\n"
            "    ports:\n"
            "      - '5432:5432'\n"
            "    volumes:\n"
            "      - ./init.sql:/docker-entrypoint-initdb.d/init.sql\n"
            "```\n\n"
            "Return findings as a JSON array with fields: issue, severity, risk, fix."
        ),
        "validator": valid_json("issue", "severity", "risk", "fix"),
    },

    # ── 6. Refactor: deeply nested conditionals → clean code ───────────────────
    {
        "id": "refactor_clean_code",
        "prompt": (
            "Refactor the function below following Clean Code principles: reduce nesting, "
            "apply early returns (guard clauses), extract helper functions where appropriate, "
            "and improve variable naming. The refactored version must produce identical output.\n\n"
            "```python\n"
            "def process_order(order):\n"
            "    result = {}\n"
            "    if order is not None:\n"
            "        if 'items' in order:\n"
            "            if len(order['items']) > 0:\n"
            "                total = 0\n"
            "                for item in order['items']:\n"
            "                    if item.get('quantity') is not None:\n"
            "                        if item.get('price') is not None:\n"
            "                            if item['quantity'] > 0:\n"
            "                                if item['price'] > 0:\n"
            "                                    total += item['quantity'] * item['price']\n"
            "                if order.get('discount') is not None:\n"
            "                    if order['discount'] > 0:\n"
            "                        if order['discount'] < 100:\n"
            "                            total = total * (1 - order['discount'] / 100)\n"
            "                result['total'] = round(total, 2)\n"
            "                result['item_count'] = len(order['items'])\n"
            "                result['status'] = 'processed'\n"
            "            else:\n"
            "                result['status'] = 'empty'\n"
            "        else:\n"
            "            result['status'] = 'invalid'\n"
            "    else:\n"
            "        result['status'] = 'null_order'\n"
            "    return result\n"
            "```\n\n"
            "Provide the refactored code in a Python code block, then explain each Clean Code "
            "technique you applied."
        ),
        "validator": run_python(
            ("process_order(None)",                                                  {"status": "null_order"}),
            ("process_order({'items': []})",                                         {"status": "empty"}),
            ("process_order({'items': [{'quantity':2,'price':15.0}]})",              {"total": 30.0, "item_count": 1, "status": "processed"}),
            ("process_order({'items': [{'quantity':2,'price':10.0}], 'discount':10})", {"total": 18.0, "item_count": 1, "status": "processed"}),
        ),
    },
]

print(f"✓ Use case: '{USE_CASE_NAME}'")
print(f"  {len(USE_CASE_PROMPTS)} prompts  |  bug detection, SQLi audit, unit tests, complexity, infra security, refactor")

### UC-C · Databricks / Data Engineering

In [ ]:
USE_CASE_NAME = "Databricks & Data Engineering"
USE_CASE_DESC = (
    "Generate PySpark transformations, write Delta Live Tables pipelines, "
    "optimize Spark SQL queries, document schemas, debug errors, and build data quality checks."
)

USE_CASE_PROMPTS = [
    # ── 1. PySpark transformation from business spec ────────────────────────────
    {
        "id": "pyspark_transformation",
        "prompt": (
            "Write a PySpark function that implements the following business transformation. "
            "Use the DataFrame API (not RDD). Include all necessary imports.\n\n"
            "Business requirement:\n"
            "  Input: a Delta table 'sales' with columns: order_id (string), customer_id (string), "
            "  product_id (string), quantity (int), unit_price (double), order_date (date), "
            "  region (string), is_returned (boolean).\n\n"
            "  Transform to produce a customer-level summary with:\n"
            "  - customer_id\n"
            "  - total_orders: count of distinct order_id where is_returned = false\n"
            "  - total_revenue: sum of (quantity * unit_price) where is_returned = false\n"
            "  - avg_order_value: total_revenue / total_orders\n"
            "  - first_order_date, last_order_date\n"
            "  - days_as_customer: days between first and last order (use datediff)\n"
            "  - top_region: the region with the highest revenue for this customer\n"
            "  - customer_segment: 'VIP' if total_revenue > 10000, 'Regular' if > 1000, else 'Occasional'\n\n"
            "Provide the complete PySpark code in a single code block."
        ),
        "validator": contains_all(
            "from pyspark", "groupBy", "customer_id", "datediff", "customer_segment", "when"
        ),
    },

    # ── 2. Delta Live Tables pipeline for medallion architecture ────────────────
    {
        "id": "dlt_pipeline",
        "prompt": (
            "Write a complete Delta Live Tables (DLT) Python pipeline for the following medallion architecture. "
            "Include @dlt.table and @dlt.expect_or_drop decorators for data quality. "
            "Add appropriate comments for each layer.\n\n"
            "Source: a streaming Kafka topic 'raw_clickstream' with JSON records:\n"
            "  {event_id, user_id, session_id, page_url, action_type, timestamp_ms, device_type, country}\n\n"
            "Bronze layer:\n"
            "  - Ingest raw JSON, parse timestamp_ms to timestamp, add ingestion_time column\n"
            "  - Quality constraint: drop records where event_id IS NULL\n\n"
            "Silver layer:\n"
            "  - Filter to only action_type IN ('click', 'view', 'purchase')\n"
            "  - Deduplicate on event_id (keep latest by timestamp)\n"
            "  - Add session_duration_seconds (seconds from first to last event in a session)\n"
            "  - Quality constraint: quarantine (not drop) records where user_id IS NULL\n\n"
            "Gold layer:\n"
            "  - Daily aggregate table: date, country, device_type, action_type → event_count, unique_users, unique_sessions\n\n"
            "Output the complete pipeline code."
        ),
        "validator": contains_all(
            "@dlt.table", "bronze", "silver", "gold", "expect_or_drop", "streaming"
        ),
    },

    # ── 3. Spark SQL query optimisation ────────────────────────────────────────
    {
        "id": "spark_query_optimisation",
        "prompt": (
            "The Spark SQL query below is running slowly on a 500 million row Delta table in Databricks. "
            "Identify ALL performance problems, explain why each is slow, and provide an optimised version. "
            "Also specify which Databricks-specific features (e.g. Z-ORDER, liquid clustering, adaptive query "
            "execution, broadcast hints) you would apply and why.\n\n"
            "```sql\n"
            "SELECT\n"
            "    c.customer_name,\n"
            "    c.email,\n"
            "    c.country,\n"
            "    SUM(o.quantity * o.unit_price) AS total_spent,\n"
            "    COUNT(o.order_id) AS order_count,\n"
            "    AVG(o.quantity * o.unit_price) AS avg_order,\n"
            "    MAX(o.order_date) AS last_order_date\n"
            "FROM orders o\n"
            "JOIN customers c ON CAST(o.customer_id AS STRING) = CAST(c.id AS INT)\n"
            "LEFT JOIN (\n"
            "    SELECT order_id, COUNT(*) AS item_count\n"
            "    FROM order_items\n"
            "    GROUP BY order_id\n"
            ") sub ON o.order_id = sub.order_id\n"
            "WHERE YEAR(o.order_date) = 2023\n"
            "  AND UPPER(c.country) IN ('US', 'UK', 'DE')\n"
            "  AND o.status NOT IN ('CANCELLED', 'RETURNED')\n"
            "GROUP BY c.customer_name, c.email, c.country\n"
            "HAVING total_spent > 500\n"
            "ORDER BY total_spent DESC\n"
            "```\n\n"
            "List each issue with its impact, then provide the optimised query."
        ),
        "validator": contains_all(
            "cast", "partition", "z-order", "broadcast", "predicate pushdown"
        ),
    },

    # ── 4. Schema documentation → data dictionary ──────────────────────────────
    {
        "id": "schema_documentation",
        "prompt": (
            "Generate a comprehensive data dictionary as a JSON array for the schema below. "
            "For each field include: column_name, data_type, nullable, description (business meaning), "
            "example_value, pii_flag (true/false), and data_quality_rule (the check you would apply).\n\n"
            "Schema (from DESCRIBE TABLE result):\n"
            "  order_id          STRING    NOT NULL   -- UUID primary key\n"
            "  customer_id       STRING    NOT NULL   -- FK to customers table\n"
            "  email             STRING    NULLABLE   -- customer email at time of order\n"
            "  order_date        DATE      NOT NULL   -- date order was placed\n"
            "  ship_date         DATE      NULLABLE   -- date shipped, null if not yet shipped\n"
            "  status            STRING    NOT NULL   -- PENDING/PROCESSING/SHIPPED/DELIVERED/CANCELLED\n"
            "  subtotal_usd      DOUBLE    NOT NULL   -- pre-discount, pre-tax total\n"
            "  discount_pct      DOUBLE    NULLABLE   -- 0.0-100.0, null means no discount\n"
            "  tax_usd           DOUBLE    NOT NULL   -- tax amount applied\n"
            "  total_usd         DOUBLE    NOT NULL   -- final amount charged\n"
            "  payment_method    STRING    NOT NULL   -- CARD/PAYPAL/CRYPTO/INVOICE\n"
            "  card_last4        STRING    NULLABLE   -- last 4 digits of card, null for non-card payments\n"
            "  ip_address        STRING    NULLABLE   -- IP at time of checkout\n"
            "  warehouse_id      INT       NOT NULL   -- FK to warehouses table\n"
            "  carrier           STRING    NULLABLE   -- FedEx/UPS/DHL/USPS\n"
            "  tracking_number   STRING    NULLABLE   -- carrier tracking number\n\n"
            "Return ONLY the JSON array."
        ),
        "validator": valid_json(
            "column_name", "data_type", "description", "pii_flag", "data_quality_rule"
        ),
    },

    # ── 5. PySpark error debugging ─────────────────────────────────────────────
    {
        "id": "spark_error_debug",
        "prompt": (
            "Debug the PySpark code below. It is failing with the error shown. "
            "Explain the root cause of each error, then provide the fully corrected code.\n\n"
            "Error 1: AnalysisException: Resolved attribute(s) 'revenue' missing from child struct\n"
            "Error 2: Py4JJavaError: Task not serializable — lambda function cannot be serialized\n"
            "Error 3: SparkException: Job aborted due to stage failure — data skew on customer_id join\n\n"
            "```python\n"
            "from pyspark.sql import SparkSession\n"
            "from pyspark.sql import functions as F\n\n"
            "spark = SparkSession.builder.appName('RevenueReport').getOrCreate()\n\n"
            "orders = spark.read.format('delta').load('/data/orders')\n"
            "customers = spark.read.format('delta').load('/data/customers')\n\n"
            "# Error 1: column alias used in same SELECT\n"
            "result = orders.select(\n"
            "    F.col('order_id'),\n"
            "    (F.col('quantity') * F.col('unit_price')).alias('revenue'),\n"
            "    (F.col('revenue') * 0.1).alias('commission')   # <-- problem\n"
            ")\n\n"
            "# Error 2: lambda in UDF registration\n"
            "multiply_udf = spark.udf.register('multiply', lambda x, y: x * y)\n\n"
            "# Error 3: join causing skew\n"
            "joined = orders.join(customers, orders.customer_id == customers.id)\n"
            "```\n\n"
            "Provide the corrected code with comments explaining each fix."
        ),
        "validator": contains_all(
            "alias", "udf", "skew", "hint", "broadcast"
        ),
    },

    # ── 6. Data quality framework ──────────────────────────────────────────────
    {
        "id": "data_quality_framework",
        "prompt": (
            "Design and implement a reusable PySpark data quality framework as a Python class. "
            "The class should:\n"
            "  1. Accept a Spark DataFrame and a list of rule definitions\n"
            "  2. Support rule types: not_null, unique, range_check(min,max), regex_match, "
            "     referential_integrity(lookup_df, key), custom_sql\n"
            "  3. Return a summary DataFrame with columns: rule_name, column, rule_type, "
            "     pass_count, fail_count, pass_rate_pct, sample_failures (array of up to 5 bad values)\n"
            "  4. Support a quarantine_mode: if True, split the input DF into (clean_df, quarantine_df)\n"
            "  5. Include a generate_report() method that prints a formatted ASCII summary table\n\n"
            "The framework must work in a Databricks notebook environment. "
            "Include a usage example at the bottom demonstrating all rule types."
        ),
        "validator": contains_all(
            "class", "def run", "not_null", "quarantine", "pass_rate", "generate_report"
        ),
    },
]

print(f"✓ Use case: '{USE_CASE_NAME}'")
print(f"  {len(USE_CASE_PROMPTS)} prompts  |  PySpark, DLT pipeline, SQL optimisation, schema docs, debugging, DQ framework")

### UC-D · Legal & Compliance Document Analysis

In [ ]:
USE_CASE_NAME = "Legal & Compliance Document Analysis"
USE_CASE_DESC = (
    "Extract structured data from contracts, identify liability clauses, flag GDPR obligations, "
    "risk-rate terms, summarise obligations, and detect non-standard provisions."
)

USE_CASE_PROMPTS = [
    # ── 1. Contract party & key date extraction ─────────────────────────────────
    {
        "id": "contract_extraction",
        "prompt": (
            "Extract all key entities and dates from the contract excerpt below. "
            "Return as JSON with fields: parties (array of {name, role, jurisdiction}), "
            "effective_date, expiry_date, notice_period_days, governing_law, "
            "renewal_type (auto/manual/none), and contract_value_usd (null if not stated).\n\n"
            "--- CONTRACT EXCERPT ---\n"
            "MASTER SERVICES AGREEMENT\n\n"
            "This Master Services Agreement ('Agreement') is entered into as of March 1, 2024 "
            "('Effective Date') between Nexus Technologies Inc., a Delaware corporation "
            "('Service Provider'), and GlobalRetail PLC, a company incorporated in England and Wales "
            "('Client').\n\n"
            "1. TERM. This Agreement shall commence on the Effective Date and continue for a period "
            "of thirty-six (36) months, unless terminated earlier pursuant to Section 12. The Agreement "
            "shall automatically renew for successive twelve (12) month periods unless either party "
            "provides written notice of non-renewal at least ninety (90) days prior to expiration.\n\n"
            "2. GOVERNING LAW. This Agreement shall be governed by the laws of the State of New York, "
            "without regard to its conflict of law provisions.\n\n"
            "3. FEES. The Client shall pay the Service Provider a monthly retainer of USD 42,000, "
            "totalling USD 1,512,000 over the initial term.\n"
            "---\n\n"
            "Return ONLY the JSON object."
        ),
        "validator": valid_json(
            "parties", "effective_date", "expiry_date", "notice_period_days",
            "governing_law", "renewal_type", "contract_value_usd"
        ),
    },

    # ── 2. Liability clause risk rating ────────────────────────────────────────
    {
        "id": "liability_risk_rating",
        "prompt": (
            "Review the liability and indemnification clauses below. For each clause:\n"
            "  (a) Assign a risk rating: Critical / High / Medium / Low\n"
            "  (b) Identify which party bears more risk\n"
            "  (c) Flag if the clause is non-standard or unusually broad/narrow\n"
            "  (d) Suggest a negotiation point if the risk is High or Critical\n\n"
            "Return as a JSON array with fields: clause_id, risk_rating, risk_bearer, "
            "is_non_standard (bool), negotiation_point.\n\n"
            "--- CLAUSES ---\n"
            "Clause A (Liability Cap): 'The total aggregate liability of either party under this "
            "Agreement shall not exceed the greater of (i) USD 500 or (ii) the fees paid in the "
            "preceding thirty (30) days.'\n\n"
            "Clause B (Indemnification): 'Client shall indemnify, defend, and hold harmless Service "
            "Provider from any and all claims, damages, losses, costs, and expenses (including "
            "reasonable attorneys' fees) arising out of or related to Client's use of the Services, "
            "whether or not caused by the negligence of Service Provider.'\n\n"
            "Clause C (Consequential Damages): 'Neither party shall be liable for any indirect, "
            "incidental, special, consequential, or punitive damages, even if advised of the "
            "possibility of such damages, provided that this limitation shall not apply to breaches "
            "of confidentiality obligations or willful misconduct.'\n\n"
            "Clause D (IP Ownership): 'All work product, deliverables, and intellectual property "
            "created by Service Provider in connection with this Agreement shall become the sole "
            "property of Client upon full payment of all fees.'\n"
            "---"
        ),
        "validator": valid_json(
            "clause_id", "risk_rating", "risk_bearer", "is_non_standard", "negotiation_point"
        ),
    },

    # ── 3. GDPR obligation mapping ──────────────────────────────────────────────
    {
        "id": "gdpr_obligation_mapping",
        "prompt": (
            "Analyse the data processing addendum (DPA) excerpt below against GDPR requirements. "
            "For each GDPR obligation listed, state whether the DPA (a) Fully Addresses, "
            "(b) Partially Addresses, or (c) Does Not Address the obligation. "
            "For any gaps, specify what language should be added.\n\n"
            "GDPR obligations to check:\n"
            "  1. Article 28(3)(a) — Processor acts only on documented instructions\n"
            "  2. Article 28(3)(b) — Confidentiality obligations on authorised persons\n"
            "  3. Article 28(3)(c) — Technical and organisational security measures\n"
            "  4. Article 28(3)(d) — Sub-processor restrictions\n"
            "  5. Article 28(3)(e) — Assistance with data subject rights\n"
            "  6. Article 28(3)(f) — Deletion or return of data at end of contract\n"
            "  7. Article 28(3)(g) — Audit rights for the controller\n"
            "  8. Article 32 — Security of processing (encryption, pseudonymisation)\n\n"
            "--- DPA EXCERPT ---\n"
            "The Processor shall: (i) process Personal Data only on written instructions from the "
            "Controller; (ii) ensure that persons authorised to process Personal Data are bound by "
            "appropriate confidentiality obligations; (iii) implement reasonable security measures "
            "to protect Personal Data; (iv) not engage sub-processors without prior written consent "
            "of the Controller; (v) upon termination, delete all Personal Data within 30 days unless "
            "required by law to retain it. The Processor shall maintain records of processing "
            "activities and make them available upon request.\n"
            "---\n\n"
            "Return as a JSON array with fields: article, obligation_summary, status, gap_description."
        ),
        "validator": valid_json(
            "article", "obligation_summary", "status", "gap_description"
        ),
    },

    # ── 4. NDA non-standard clause detection ───────────────────────────────────
    {
        "id": "nda_clause_review",
        "prompt": (
            "Review the NDA clauses below and identify any provisions that deviate significantly "
            "from market-standard mutual NDAs. For each non-standard finding: explain why it is "
            "unusual, what risk it creates, and suggest standard alternative language.\n\n"
            "--- NDA CLAUSES ---\n"
            "1. CONFIDENTIALITY PERIOD: 'The confidentiality obligations under this Agreement shall "
            "survive in perpetuity and shall remain binding on both parties forever.'\n\n"
            "2. DEFINITION OF CONFIDENTIAL INFORMATION: 'Confidential Information means any and all "
            "information disclosed by either party, regardless of whether it is marked confidential, "
            "including but not limited to information that is generally known to the public.'\n\n"
            "3. RESIDUALS: 'Notwithstanding any other provision, the Receiving Party shall have no "
            "obligation to restrict its personnel who, without reference to written materials, "
            "retain in their unaided memories information learned during the course of performing "
            "services under this Agreement.'\n\n"
            "4. REMEDY: 'The parties agree that monetary damages would be an adequate remedy for "
            "any breach of this Agreement and that injunctive relief shall not be available.'\n\n"
            "5. EXCLUSIONS: 'Information shall not be deemed Confidential Information if it: "
            "(a) was in the public domain prior to disclosure; (b) becomes public domain through "
            "no fault of the Receiving Party; (c) was independently developed.'\n"
            "---\n\n"
            "Provide your analysis as a JSON array with fields: clause_number, is_non_standard, "
            "deviation_description, risk, suggested_language."
        ),
        "validator": valid_json(
            "clause_number", "is_non_standard", "deviation_description", "risk", "suggested_language"
        ),
    },

    # ── 5. Contract obligation timeline extraction ──────────────────────────────
    {
        "id": "obligation_timeline",
        "prompt": (
            "Extract all time-bound obligations from the contract clauses below. "
            "For each obligation: identify the responsible party, the action required, "
            "the deadline or trigger event, and consequence of non-compliance.\n"
            "Return as a JSON array sorted by urgency (shortest deadline first), "
            "with fields: party, action, deadline_or_trigger, consequence, urgency_rank.\n\n"
            "--- CONTRACT CLAUSES ---\n"
            "Section 5.1: Service Provider shall deliver the initial project plan within ten (10) "
            "business days of the Effective Date.\n\n"
            "Section 5.3: Client shall provide written approval or detailed written objections "
            "within five (5) business days of receiving each deliverable. Failure to respond "
            "shall be deemed acceptance.\n\n"
            "Section 7.2: Either party may terminate this Agreement for cause upon thirty (30) "
            "days' written notice if the other party materially breaches this Agreement and fails "
            "to cure such breach within the notice period.\n\n"
            "Section 9.1: Service Provider shall notify Client of any confirmed data breach "
            "within forty-eight (48) hours of discovery.\n\n"
            "Section 11.4: All invoices are due and payable within thirty (30) days of the invoice "
            "date. Late payments accrue interest at 1.5% per month.\n\n"
            "Section 14.2: Upon expiration or termination, Service Provider shall return or destroy "
            "all Client Confidential Information within fifteen (15) business days and provide "
            "written certification thereof.\n"
            "---"
        ),
        "validator": valid_json(
            "party", "action", "deadline_or_trigger", "consequence", "urgency_rank"
        ),
    },

    # ── 6. Regulatory compliance gap analysis ──────────────────────────────────
    {
        "id": "compliance_gap_analysis",
        "prompt": (
            "A fintech startup wants to launch a consumer lending product in the United States. "
            "Based on the product description below, perform a regulatory compliance gap analysis. "
            "Identify which federal and state regulations apply, what specific compliance requirements "
            "exist, whether the described practices are compliant, and the risk of non-compliance.\n\n"
            "--- PRODUCT DESCRIPTION ---\n"
            "Product: 'QuickCash' — an unsecured personal loan product.\n"
            "  - Loan amounts: $500 to $5,000\n"
            "  - APR range: 28% to 199% depending on credit score\n"
            "  - Loan term: 3 to 24 months\n"
            "  - Origination fee: 5% of loan amount, deducted upfront from disbursement\n"
            "  - Late fee: $35 or 5% of the payment, whichever is greater\n"
            "  - Marketing: targeted ads shown to users who have searched for 'payday loan' or "
            "    'emergency cash', including users aged 18-21\n"
            "  - Application: fully digital, with ID verification via selfie + government ID\n"
            "  - Credit check: soft pull only, no hard inquiry\n"
            "  - Collections: automated SMS and email starting 1 day after missed payment\n"
            "  - Data sharing: loan performance data shared with two credit bureaus\n"
            "---\n\n"
            "Structure your response as a JSON array with fields: regulation, requirement, "
            "current_practice_assessment (Compliant/Non-Compliant/Unclear), risk_level, "
            "recommended_action."
        ),
        "validator": valid_json(
            "regulation", "requirement", "current_practice_assessment",
            "risk_level", "recommended_action"
        ),
    },
]

print(f"✓ Use case: '{USE_CASE_NAME}'")
print(f"  {len(USE_CASE_PROMPTS)} prompts  |  contract extraction, liability risk, GDPR mapping, NDA review, obligations, compliance")

---
## Cell 5 · ⚖️ Set recommendation weights  ← TUNE FOR YOUR PRIORITIES

In [ ]:
# Preset profiles — uncomment one or set custom weights below
# PROFILE = "balanced"          # equal weight to all three
# PROFILE = "quality_first"     # accuracy matters most
# PROFILE = "cost_sensitive"    # cost matters most
# PROFILE = "latency_critical"  # speed matters most
PROFILE = "balanced"

WEIGHT_PRESETS = {
    "balanced":         {"accuracy": 0.40, "cost": 0.30, "speed": 0.30},
    "quality_first":    {"accuracy": 0.60, "cost": 0.20, "speed": 0.20},
    "cost_sensitive":   {"accuracy": 0.30, "cost": 0.55, "speed": 0.15},
    "latency_critical": {"accuracy": 0.30, "cost": 0.20, "speed": 0.50},
}

WEIGHTS = WEIGHT_PRESETS[PROFILE]
# OR override manually:
# WEIGHTS = {"accuracy": 0.5, "cost": 0.3, "speed": 0.2}

assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, "Weights must sum to 1.0"

print(f"✓ Profile : {PROFILE}")
print(f"  Accuracy weight : {WEIGHTS['accuracy']:.0%}")
print(f"  Cost     weight : {WEIGHTS['cost']:.0%}")
print(f"  Speed    weight : {WEIGHTS['speed']:.0%}")

---
## Cell 6 · Run benchmark against your use-case prompts

In [ ]:
def calc_cost(model_id, inp, out):
    p = PRICING.get(model_id)
    if not p: return 0.0
    return (inp/1_000_000)*p["input_price_per_1m"] + (out/1_000_000)*p["output_price_per_1m"]

def call_with_retry(model_id, prompt_text, max_tokens=512, max_retries=4, base_wait=2):
    for attempt in range(max_retries):
        try:
            return client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt_text}],
                max_tokens=max_tokens,
                temperature=0,
            )
        except openai.RateLimitError:
            wait = base_wait * (2 ** attempt)
            if attempt < max_retries - 1:
                print(f"  ⏳ rate limit — retrying in {wait}s ...", flush=True)
                time.sleep(wait)
            else:
                raise

CALL_DELAY = 1.0   # seconds between calls — raise if you hit 429s
raw_results = []
total = len(MODELS) * len(USE_CASE_PROMPTS)
count = 0

for model in MODELS:
    for prompt in USE_CASE_PROMPTS:
        count += 1
        tag = f"[{count:>2}/{total}] {model['display_name']:<24} ← {prompt['id']}"
        print(tag, end="  ", flush=True)
        start = time.perf_counter()
        try:
            resp = call_with_retry(model["model_id"], prompt["prompt"])
            elapsed = round(time.perf_counter()-start, 3)
            text    = resp.choices[0].message.content or ""
            inp     = resp.usage.prompt_tokens     if resp.usage else 0
            out     = resp.usage.completion_tokens if resp.usage else 0
            cost    = calc_cost(model["model_id"], inp, out)
            label, score, detail = prompt["validator"](text)
            print(f"→ {label:<10} {score:.2f}  {elapsed:.2f}s  ${cost:.6f}")
            raw_results.append({"model": model["display_name"], "model_id": model["model_id"],
                "provider": model["provider"], "prompt_id": prompt["id"],
                "input_tokens": inp, "output_tokens": out, "total_tokens": inp+out,
                "ai_cost_usd": cost, "response_time_s": elapsed,
                "accuracy_label": label, "accuracy_score": score,
                "accuracy_detail": detail, "response": text, "error": None})
        except Exception as e:
            elapsed = round(time.perf_counter()-start, 3)
            print(f"→ SKIP (not available)")
            raw_results.append({"model": model["display_name"], "model_id": model["model_id"],
                "provider": model["provider"], "prompt_id": prompt["id"],
                "input_tokens": 0, "output_tokens": 0, "total_tokens": 0,
                "ai_cost_usd": 0, "response_time_s": elapsed,
                "accuracy_label": "Error", "accuracy_score": 0,
                "accuracy_detail": str(e), "response": "", "error": str(e)})
        time.sleep(CALL_DELAY)

print(f"\n✓ Done — {len(raw_results)} results  |  errors: {sum(1 for r in raw_results if r['error'])}")

---
## Cell 7 · Score computation

In [ ]:
from collections import defaultdict

# Aggregate per model — skip models with all errors
agg = defaultdict(lambda: {"acc":[], "time":[], "cost":[], "provider":"", "errors":0, "calls":0})
for r in raw_results:
    a = agg[r["model"]]
    a["provider"] = r["provider"]
    a["calls"] += 1
    if r["error"]:
        a["errors"] += 1
    else:
        a["acc"].append(r["accuracy_score"])
        a["time"].append(r["response_time_s"])
        a["cost"].append(r["ai_cost_usd"])

# Build model metrics — only models with at least one successful call
metrics = {}
for name, a in agg.items():
    if not a["acc"]: continue   # all failed — skip from quadrant
    metrics[name] = {
        "provider":    a["provider"],
        "avg_accuracy": sum(a["acc"]) / len(a["acc"]),
        "avg_time_s":   sum(a["time"]) / len(a["time"]),
        "total_cost":   sum(a["cost"]),
        "errors":       a["errors"],
        "calls":        a["calls"],
    }

model_names = list(metrics.keys())

# Normalise 0-1 across all available models
def normalise(vals):
    mn, mx = min(vals), max(vals)
    return [0.5 if mx==mn else (v-mn)/(mx-mn) for v in vals]

accs   = [metrics[m]["avg_accuracy"] for m in model_names]
times  = [metrics[m]["avg_time_s"]   for m in model_names]
costs  = [metrics[m]["total_cost"]   for m in model_names]

norm_acc   = normalise(accs)
norm_speed = normalise([1/t if t>0 else 0 for t in times])   # invert: fast = high score
norm_value = normalise([1/(c+1e-9) for c in costs])           # invert: cheap = high score

# Composite recommendation score
for i, m in enumerate(model_names):
    metrics[m]["norm_accuracy"] = norm_acc[i]
    metrics[m]["norm_speed"]    = norm_speed[i]
    metrics[m]["norm_value"]    = norm_value[i]
    metrics[m]["composite"]     = (
        WEIGHTS["accuracy"] * norm_acc[i]
        + WEIGHTS["cost"]    * norm_value[i]
        + WEIGHTS["speed"]   * norm_speed[i]
    )

# Rank
ranked = sorted(model_names, key=lambda m: metrics[m]["composite"], reverse=True)
winner = ranked[0]

print(f"{'Model':<28} {'Accuracy':>9} {'Avg Time':>10} {'Total Cost':>12} {'Composite':>10}")
print("─"*75)
for m in ranked:
    x = metrics[m]
    medal = " ⭐ RECOMMENDED" if m==winner else ""
    print(f"{m:<28} {x['avg_accuracy']:>9.2f} {x['avg_time_s']:>9.2f}s ${x['total_cost']:>10.6f} {x['composite']:>10.3f}{medal}")

---
## Cell 8 · Magic Quadrant

In [ ]:
PROVIDER_COLORS = {
    "Anthropic": "#E07B39",
    "OpenAI":    "#10A37F",
    "Google":    "#4285F4",
    "Meta":      "#1877F2",
}

fig, axes = plt.subplots(1, 2, figsize=(20, 9),
                         gridspec_kw={"width_ratios": [2, 1]})
fig.patch.set_facecolor("#F4F6F9")
fig.suptitle(
    f"LLM Magic Quadrant — {USE_CASE_NAME}\n"
    f"Profile: {PROFILE.replace('_',' ').title()}  "
    f"(Accuracy {WEIGHTS['accuracy']:.0%}  ·  Cost {WEIGHTS['cost']:.0%}  ·  Speed {WEIGHTS['speed']:.0%})",
    fontsize=15, fontweight="bold", y=1.01
)

# ── Left panel: Magic Quadrant scatter ───────────────────────────────────────
ax = axes[0]
ax.set_facecolor("#FFFFFF")
ax.set_xlim(-0.05, 1.15)
ax.set_ylim(-0.05, 1.15)
ax.set_xlabel("Performance Score  (Accuracy →)", fontsize=12, labelpad=8)
ax.set_ylabel("Value Score  (Cost Efficiency →)", fontsize=12, labelpad=8)
ax.set_title("Magic Quadrant", fontsize=13, fontweight="bold", pad=10)

# Quadrant dividers at 0.5
ax.axvline(0.5, color="#BBBBBB", lw=1.5, ls="--", zorder=1)
ax.axhline(0.5, color="#BBBBBB", lw=1.5, ls="--", zorder=1)

# Quadrant background shading
quad_cfg = [
    (0.5, 0.5, 0.65, 0.65, "#E8F5E9", "⭐ LEADERS\n(Recommended)",   "#2e7d32", "center"),
    (0.0, 0.5, 0.50, 0.65, "#FFF3E0", "💰 BUDGET OPTIONS\n(Lower quality)",  "#e65100", "center"),
    (0.5, 0.0, 0.65, 0.50, "#E3F2FD", "🏆 PREMIUM QUALITY\n(Higher cost)",   "#1565c0", "center"),
    (0.0, 0.0, 0.50, 0.50, "#FFEBEE", "⚠️  AVOID\n(Costly & inaccurate)",  "#b71c1c", "center"),
]
for x0,y0,lx,ly,bg,label,tc,ha in quad_cfg:
    ax.add_patch(plt.Rectangle((x0,y0), 0.5, 0.5, color=bg, zorder=0, alpha=0.6))
    ax.text(x0+0.25, y0+0.25, label, ha="center", va="center",
            fontsize=8.5, color=tc, alpha=0.55, fontweight="bold", linespacing=1.5)

# Plot each model
used_providers = set()
for m in model_names:
    x_val  = metrics[m]["norm_accuracy"]
    y_val  = metrics[m]["norm_value"]
    speed  = metrics[m]["norm_speed"]
    prov   = metrics[m]["provider"]
    color  = PROVIDER_COLORS.get(prov, "#888888")
    size   = 300 + speed * 1200   # bubble size ∝ speed
    is_winner = (m == winner)

    ax.scatter(x_val, y_val, s=size, color=color,
               edgecolors="#FFD700" if is_winner else "white",
               linewidths=4 if is_winner else 1.5,
               zorder=5, alpha=0.92)

    # Label
    short = m.replace(" Instruct","").replace(" Maverick"," Mvk")
    offset_y = 0.07 if y_val < 0.85 else -0.07
    ax.annotate(
        ("⭐ " if is_winner else "") + short,
        xy=(x_val, y_val), xytext=(x_val, y_val + offset_y),
        ha="center", va="center", fontsize=8,
        fontweight="bold" if is_winner else "normal",
        color="#1a1a2e",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=color, alpha=0.85, lw=1.2)
    )

    # Data callout: acc / time / cost
    details = (
        f"acc={metrics[m]['avg_accuracy']:.2f}\n"
        f"{metrics[m]['avg_time_s']:.2f}s\n"
        f"${metrics[m]['total_cost']:.5f}"
    )
    ax.annotate(details, xy=(x_val, y_val),
                xytext=(x_val + 0.03, y_val - 0.11),
                fontsize=6.5, color="#555555", ha="left", va="top",
                bbox=dict(boxstyle="round,pad=0.2", fc="#F9F9F9", ec="#DDDDDD", alpha=0.8))
    used_providers.add(prov)

# Legend — provider colours
handles = [mpatches.Patch(color=PROVIDER_COLORS[p], label=p) for p in sorted(used_providers)]
handles.append(plt.scatter([], [], s=300, c="gray", label="Bubble size = Speed"))
ax.legend(handles=handles, fontsize=9, loc="lower right",
          framealpha=0.9, edgecolor="#CCCCCC")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ── Right panel: Recommendation scorecard ────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("#FFFFFF")
ax2.axis("off")
ax2.set_title("Recommendation Scorecard", fontsize=13, fontweight="bold", pad=10)

# Header
wm = metrics[winner]
ax2.text(0.5, 0.97,
    f"\n⭐  RECOMMENDED FOR\n"
    f"\"{USE_CASE_NAME}\"",
    ha="center", va="top", fontsize=11, fontweight="bold",
    color="#2e7d32", transform=ax2.transAxes, linespacing=1.6)

ax2.text(0.5, 0.80, winner,
    ha="center", va="top", fontsize=16, fontweight="bold",
    color=PROVIDER_COLORS.get(wm["provider"],"#333"),
    transform=ax2.transAxes)

ax2.add_patch(plt.Rectangle((0.05, 0.60), 0.90, 0.001,
    color="#CCCCCC", transform=ax2.transAxes))

# Metric rows
row_data = [
    ("Accuracy Score",   f"{wm['avg_accuracy']:.2f} / 1.00",    "#2e7d32"),
    ("Avg Response Time",f"{wm['avg_time_s']:.2f}s",             "#1565c0"),
    ("Total AI Cost",    f"${wm['total_cost']:.6f}",             "#6a1b9a"),
    ("Composite Score",  f"{wm['composite']:.3f} / 1.000",       "#e65100"),
    ("Provider",         wm["provider"],                          "#444444"),
    ("Profile Used",     PROFILE.replace("_"," ").title(),        "#444444"),
]
for i,(label,val,color) in enumerate(row_data):
    y = 0.55 - i*0.09
    ax2.text(0.08, y, label+":", ha="left", va="center", fontsize=9.5,
             color="#555555", transform=ax2.transAxes)
    ax2.text(0.92, y, val,      ha="right", va="center", fontsize=10,
             fontweight="bold", color=color, transform=ax2.transAxes)

# Runner-up
if len(ranked) > 1:
    runner = ranked[1]
    ax2.add_patch(plt.Rectangle((0.05, 0.01), 0.90, 0.001,
        color="#CCCCCC", transform=ax2.transAxes))
    ax2.text(0.5, 0.08,
        f"Runner-up: {runner}\n"
        f"Composite: {metrics[runner]['composite']:.3f}  "
        f"Accuracy: {metrics[runner]['avg_accuracy']:.2f}",
        ha="center", va="bottom", fontsize=8.5, color="#666666",
        transform=ax2.transAxes, linespacing=1.5)

plt.tight_layout(pad=2)
OUT = f"llm_magic_quadrant_{USE_CASE_NAME.lower().replace(' ','_')}.png"
plt.savefig(OUT, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"\n✓ Magic Quadrant saved → {OUT}")
print(f"\n🏆 Recommendation: {winner}")
print(f"   Composite score : {wm['composite']:.3f}")
print(f"   Accuracy        : {wm['avg_accuracy']:.2f}")
print(f"   Avg time        : {wm['avg_time_s']:.2f}s")
print(f"   Total cost      : ${wm['total_cost']:.6f}")
plt.show()

---
## Cell 9 · Full scorecard table

In [ ]:
try:
    import pandas as pd
    rows = []
    for rank, m in enumerate(ranked, 1):
        x = metrics[m]
        def pct(val, ref): return f"{'+'if val>=ref else ''}{((val-ref)/ref*100):.1f}%" if ref else "N/A"
        mean_acc  = sum(metrics[n]["avg_accuracy"] for n in model_names)/len(model_names)
        mean_cost = sum(metrics[n]["total_cost"]   for n in model_names)/len(model_names)
        mean_time = sum(metrics[n]["avg_time_s"]   for n in model_names)/len(model_names)
        rows.append({
            "Rank":         f"#{rank}" + (" ⭐" if m==winner else ""),
            "Model":        m,
            "Provider":     x["provider"],
            "Accuracy":     round(x["avg_accuracy"],3),
            "Acc vs Mean":  pct(x["avg_accuracy"], mean_acc),
            "Avg Time (s)": round(x["avg_time_s"],2),
            "Time vs Mean": pct(x["avg_time_s"],   mean_time),
            "Total Cost $": round(x["total_cost"],6),
            "Cost vs Mean": pct(x["total_cost"],   mean_cost),
            "Composite":    round(x["composite"],3),
            "Errors":       x["errors"],
        })

    ACCURACY_COLORS = {"Excellent":"#c6efce","Good":"#ffeb9c","Partial":"#ffcc99","Poor":"#ffc7ce"}
    df = pd.DataFrame(rows)

    def color_acc(v):
        return "color:green;font-weight:bold" if isinstance(v,str) and v.startswith("+") else \
               "color:red;font-weight:bold"   if isinstance(v,str) and v.startswith("-") else ""
    def color_cost(v):
        return "color:red;font-weight:bold"   if isinstance(v,str) and v.startswith("+") else \
               "color:green;font-weight:bold" if isinstance(v,str) and v.startswith("-") else ""

    df.style \
        .background_gradient(subset=["Accuracy"],   cmap="RdYlGn") \
        .background_gradient(subset=["Total Cost $"],cmap="RdYlGn_r") \
        .background_gradient(subset=["Avg Time (s)"],cmap="RdYlGn_r") \
        .background_gradient(subset=["Composite"],   cmap="RdYlGn") \
        .applymap(color_acc,  subset=["Acc vs Mean"]) \
        .applymap(color_cost, subset=["Time vs Mean","Cost vs Mean"]) \
        .set_caption(f"Full scorecard — {USE_CASE_NAME}  |  Profile: {PROFILE}") \
        .hide(axis="index")
except ImportError:
    print("Install pandas for the styled table: %pip install pandas")

---
## Cell 10 · Export results

In [ ]:
safe_name = USE_CASE_NAME.lower().replace(" ","_")

# Raw results CSV
try:
    import pandas as pd
    csv_path = f"llm_benchmark_{safe_name}.csv"
    pd.DataFrame([{k:v for k,v in r.items() if k!="response"} for r in raw_results]) \
      .to_csv(csv_path, index=False)
    print(f"✓ Raw results  → {csv_path}")
except ImportError:
    pass

# Scorecard JSON
sc_path = f"llm_scorecard_{safe_name}.json"
with open(sc_path, "w") as f:
    json.dump({
        "use_case": USE_CASE_NAME,
        "profile":  PROFILE,
        "weights":  WEIGHTS,
        "recommendation": winner,
        "scores": {m: {
            "accuracy":   round(metrics[m]["avg_accuracy"],4),
            "avg_time_s": round(metrics[m]["avg_time_s"],3),
            "total_cost": round(metrics[m]["total_cost"],8),
            "composite":  round(metrics[m]["composite"],4),
        } for m in ranked}
    }, f, indent=2)
print(f"✓ Scorecard JSON → {sc_path}")
print(f"✓ Quadrant image → llm_magic_quadrant_{safe_name}.png")